# GAN vs WGAN: A Simple Comparison on MNIST

This notebook implements both a standard **GAN** (Generative Adversarial Network) and a **WGAN** (Wasserstein GAN) on MNIST so you can see the key differences side by side.

## 1. The Core Difference: Loss Functions

### Standard GAN (Binary Cross-Entropy Loss)

The discriminator outputs a probability (via sigmoid) and is trained with **binary cross-entropy**:

$$\mathcal{L}_D = -\mathbb{E}_{x \sim p_{data}}[\log D(x)] - \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

$$\mathcal{L}_G = -\mathbb{E}_{z \sim p_z}[\log D(G(z))]$$

**Problem:** When the discriminator gets too good, the gradient for the generator vanishes. This makes training unstable and prone to **mode collapse** (the generator only learns to produce a few types of outputs).

### WGAN (Wasserstein Loss)

The critic (no longer called discriminator) outputs an **unbounded real number** (no sigmoid) and is trained with the **Wasserstein distance**:

$$\mathcal{L}_C = \mathbb{E}_{z \sim p_z}[C(G(z))] - \mathbb{E}_{x \sim p_{data}}[C(x)]$$

$$\mathcal{L}_G = -\mathbb{E}_{z \sim p_z}[C(G(z))]$$

**Advantage:** The Wasserstein loss provides meaningful gradients even when the critic is well-trained, leading to more stable training and less mode collapse.

## 2. Key Architectural Differences at a Glance

| Aspect | GAN | WGAN |
|---|---|---|
| Discriminator output | Sigmoid (probability 0-1) | Raw score (no sigmoid) |
| Loss function | Binary Cross-Entropy | Wasserstein distance |
| Discriminator name | Discriminator | Critic |
| Optimizer | Adam works well | **RMSProp** (or Adam with low beta1) |
| Critic training | 1 step per generator step | **Multiple steps** (e.g. 5) per generator step |
| Constraint | None | Weight clipping (or gradient penalty in WGAN-GP) |
| Training stability | Can be unstable | More stable |

## 3. Setup

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
LATENT_DIM = 64
BATCH_SIZE = 64
IMG_DIM = 28 * 28  # MNIST images flattened
EPOCHS = 50
LR = 0.0002

# MNIST dataloader
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

Using device: cuda


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.04MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.5MB/s]


## 4. Shared Generator Architecture

Both GAN and WGAN use the **exact same generator**. The difference is only in the discriminator/critic and the loss.

In [3]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, IMG_DIM),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)

## 5. Standard GAN

Notice the **sigmoid** at the end of the discriminator and the use of **BCELoss**.

In [4]:
class GANDiscriminator(nn.Module):
    """Outputs a probability via sigmoid."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(IMG_DIM, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),  # <-- KEY DIFFERENCE: outputs probability
        )

    def forward(self, x):
        return self.net(x)


def train_gan(dataloader, epochs):
    G = Generator().to(device)
    D = GANDiscriminator().to(device)
    opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
    criterion = nn.BCELoss()  # <-- Binary Cross-Entropy

    g_losses, d_losses = [], []

    for epoch in range(epochs):
        for real_imgs, _ in dataloader:
            real_imgs = real_imgs.view(-1, IMG_DIM).to(device)
            batch_size = real_imgs.size(0)
            real_labels = torch.ones(batch_size, 1, device=device)
            fake_labels = torch.zeros(batch_size, 1, device=device)

            # --- Train Discriminator ---
            z = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_imgs = G(z).detach()

            loss_real = criterion(D(real_imgs), real_labels)
            loss_fake = criterion(D(fake_imgs), fake_labels)
            d_loss = loss_real + loss_fake

            opt_D.zero_grad()
            d_loss.backward()
            opt_D.step()

            # --- Train Generator ---
            z = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_imgs = G(z)
            g_loss = criterion(D(fake_imgs), real_labels)  # fool the discriminator

            opt_G.zero_grad()
            g_loss.backward()
            opt_G.step()

        g_losses.append(g_loss.item())
        d_losses.append(d_loss.item())
        if (epoch + 1) % 10 == 0:
            print(f"[GAN] Epoch {epoch+1}/{epochs}  D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return G, g_losses, d_losses

## 6. Wasserstein GAN (WGAN)

Key differences from the standard GAN:
1. **No sigmoid** in the critic (raw score output)
2. **Wasserstein loss** instead of BCE (just the mean score difference)
3. **Weight clipping** to enforce the Lipschitz constraint
4. **Train the critic more often** (5 steps for every 1 generator step)
5. **RMSProp** optimizer instead of Adam

In [5]:
class WGANCritic(nn.Module):
    """Outputs an unbounded score (no sigmoid)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(IMG_DIM, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),  # <-- NO sigmoid!
        )

    def forward(self, x):
        return self.net(x)


CRITIC_ITERS = 5   # train critic 5x more than generator
WEIGHT_CLIP = 0.01  # clip critic weights to [-c, c]


def train_wgan(dataloader, epochs):
    G = Generator().to(device)
    C = WGANCritic().to(device)
    opt_G = optim.RMSprop(G.parameters(), lr=5e-5)  # <-- RMSProp, not Adam
    opt_C = optim.RMSprop(C.parameters(), lr=5e-5)

    g_losses, c_losses = [], []

    for epoch in range(epochs):
        data_iter = iter(dataloader)
        i = 0
        while i < len(dataloader):
            # --- Train Critic for CRITIC_ITERS steps ---
            for _ in range(CRITIC_ITERS):
                if i >= len(dataloader):
                    break
                real_imgs, _ = next(data_iter)
                i += 1
                real_imgs = real_imgs.view(-1, IMG_DIM).to(device)
                batch_size = real_imgs.size(0)

                z = torch.randn(batch_size, LATENT_DIM, device=device)
                fake_imgs = G(z).detach()

                # Wasserstein loss: maximize E[C(real)] - E[C(fake)]
                # Equivalently, minimize E[C(fake)] - E[C(real)]
                c_loss = C(fake_imgs).mean() - C(real_imgs).mean()

                opt_C.zero_grad()
                c_loss.backward()
                opt_C.step()

                # Weight clipping to enforce Lipschitz constraint
                for p in C.parameters():
                    p.data.clamp_(-WEIGHT_CLIP, WEIGHT_CLIP)

            # --- Train Generator ---
            z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
            fake_imgs = G(z)
            g_loss = -C(fake_imgs).mean()  # maximize critic score on fakes

            opt_G.zero_grad()
            g_loss.backward()
            opt_G.step()

        g_losses.append(g_loss.item())
        c_losses.append(c_loss.item())
        print(f"[WGAN] Epoch {epoch+1}/{epochs}  C_loss: {c_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return G, g_losses, c_losses

## 7. Train Both Models

Let's train both and compare. This will take a few minutes depending on your hardware.

In [5]:
print("=" * 50)
print("Training Standard GAN...")
print("=" * 50)
gan_G, gan_g_losses, gan_d_losses = train_gan(dataloader, EPOCHS)

Training Standard GAN...
[GAN] Epoch 10/50  D_loss: 0.8359  G_loss: 1.1695
[GAN] Epoch 20/50  D_loss: 1.1136  G_loss: 0.6787
[GAN] Epoch 30/50  D_loss: 1.0040  G_loss: 1.7053
[GAN] Epoch 40/50  D_loss: 1.0117  G_loss: 1.0322
[GAN] Epoch 50/50  D_loss: 0.8263  G_loss: 1.4517


In [ ]:
print("=" * 50)
print("Training WGAN...")
print("=" * 50)
wgan_G, wgan_g_losses, wgan_c_losses = train_wgan(dataloader, EPOCHS)

Training WGAN...
[WGAN] Epoch 1/50  C_loss: -0.1351  G_loss: -12.7115
[WGAN] Epoch 2/50  C_loss: -0.0657  G_loss: -4.0159
[WGAN] Epoch 3/50  C_loss: -0.3503  G_loss: -2.2694
[WGAN] Epoch 4/50  C_loss: -0.4163  G_loss: -4.1939
[WGAN] Epoch 5/50  C_loss: -0.3049  G_loss: -3.3067
[WGAN] Epoch 6/50  C_loss: -0.3144  G_loss: -2.5919
[WGAN] Epoch 7/50  C_loss: -0.4102  G_loss: -1.5089
[WGAN] Epoch 8/50  C_loss: -0.3043  G_loss: -1.5666


## 8. Compare Loss Curves

One of the biggest practical advantages of WGAN is that the **critic loss correlates with sample quality**. In standard GAN, the discriminator loss is often meaningless once it saturates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].set_title("Standard GAN Losses")
axes[0].plot(gan_g_losses, label="Generator")
axes[0].plot(gan_d_losses, label="Discriminator")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title("WGAN Losses")
axes[1].plot(wgan_g_losses, label="Generator")
axes[1].plot(wgan_c_losses, label="Critic")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Compare Generated Samples

In [ ]:
def show_samples(generator, title, n=16):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n, LATENT_DIM, device=device)
        samples = generator(z).cpu().view(n, 1, 28, 28)
        samples = (samples + 1) / 2  # rescale from [-1, 1] to [0, 1]

    fig, axes = plt.subplots(2, n // 2, figsize=(n, 4))
    fig.suptitle(title, fontsize=14)
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i].squeeze(), cmap="gray")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(gan_G, "Standard GAN - Generated Digits")
show_samples(wgan_G, "WGAN - Generated Digits")